# 15 注文明細と価格・キャンペーンの整合性

全10店舗の注文明細を店舗ごとに読み、同じ日・店舗・商品に有効な価格、適用されたキャンペーンの日付・店舗・対象カテゴリを照合する。数量×定価と値引き後金額の算術も確認する。結果は配布データの観察であり、コンペのルールを推測して変更するものではない。

`store_ids` の `all` と `|` 区切りを展開し、有効期間は両端を含む。2025年の来店・注文実績は存在しないため、この分析は2022〜2024年に限られる。

In [1]:
from pathlib import Path
import zipfile,io
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
ROOT=Path.cwd()
if not (ROOT/'input').is_dir(): ROOT=ROOT.parent
assert (ROOT/'input').is_dir()
INPUT=ROOT/'input';pd.set_option('display.max_columns',40);pd.options.display.float_format='{:,.4f}'.format
stores=pd.read_csv(INPUT/'stores.csv').store_id.tolist()
menus=pd.read_csv(INPUT/'menus.csv')[['item_id','category']]
prices=pd.read_csv(INPUT/'price_history.csv',parse_dates=['effective_start_date','effective_end_date'])
campaigns=pd.read_csv(INPUT/'campaign_calendar.csv',parse_dates=['date'])

def expand_stores(frame):
    x=frame.copy();x['store_id']=x.store_ids.map(lambda s:stores if s=='all' else str(s).split('|'))
    return x.explode('store_id',ignore_index=True)
prices=expand_stores(prices);campaigns=expand_stores(campaigns)
assert not campaigns.duplicated(['date','store_id','campaign_id']).any()
assert prices.store_id.isin(stores).all() and campaigns.store_id.isin(stores).all()
print('価格履歴行（店舗展開後）:',len(prices),'キャンペーン行（店舗展開後）:',len(campaigns))

価格履歴行（店舗展開後）: 2520 キャンペーン行（店舗展開後）: 22338


## 価格・割引の全件照合
価格履歴は店舗ごとに日×商品へ展開する。クーポンがある行だけ、企画ID・日付・店舗と対象カテゴリを照合する。合計値の一致では行単位の矛盾を見逃すため、全行で条件を数える。

In [2]:
summaries=[]; coupon_summaries=[]; examples=[]
cols=['order_id','date','item_id','quantity','list_price_amount','discount_amount','net_sales_amount','applied_coupon_campaign_id']
with zipfile.ZipFile(INPUT/'logs.zip') as z:
    for sid in stores:
        with z.open(f'logs/{sid}/order_items.parquet') as f:
            o=pq.read_table(io.BytesIO(f.read()),columns=cols).to_pandas()
        p=prices[prices.store_id.eq(sid)].copy()
        p['date']=[pd.date_range(a,b) for a,b in zip(p.effective_start_date,p.effective_end_date)]
        p=p.explode('date',ignore_index=True)[['date','item_id','selling_price']]
        assert not p.duplicated(['date','item_id']).any()
        o=o.merge(p,on=['date','item_id'],how='left',validate='many_to_one')
        o=o.merge(menus,on='item_id',how='left',validate='many_to_one')
        o['year']=o.date.dt.year
        o['missing_price']=o.selling_price.isna();o['missing_menu']=o.category.isna()
        o['bad_list_price']=~o.missing_price & (o.list_price_amount!=o.quantity*o.selling_price)
        o['bad_net']=o.net_sales_amount.ne(o.list_price_amount-o.discount_amount)
        o['discount_without_coupon']=o.applied_coupon_campaign_id.isna() & o.discount_amount.ne(0)
        o['coupon_without_discount']=o.applied_coupon_campaign_id.notna() & o.discount_amount.eq(0)
        o['negative_amount']=(o.quantity<=0)|(o.list_price_amount<0)|(o.discount_amount<0)|(o.net_sales_amount<0)
        agg=o.groupby('year',observed=True).agg(lines=('item_id','size'),missing_price=('missing_price','sum'),missing_menu=('missing_menu','sum'),bad_list_price=('bad_list_price','sum'),bad_net=('bad_net','sum'),discount_without_coupon=('discount_without_coupon','sum'),coupon_without_discount=('coupon_without_discount','sum'),negative_amount=('negative_amount','sum')).reset_index();agg['store_id']=sid;summaries.append(agg)
        issue=o[o[['missing_price','missing_menu','bad_list_price','bad_net','discount_without_coupon','coupon_without_discount','negative_amount']].any(axis=1)].head(3)
        if len(issue): examples.append(issue.assign(store_id=sid))
        cp=o[o.applied_coupon_campaign_id.notna()].copy()
        c=campaigns[campaigns.store_id.eq(sid)][['date','campaign_id','campaign_type','coupon_flag','discount_rate','target_item_group']]
        cp=cp.merge(c,left_on=['date','applied_coupon_campaign_id'],right_on=['date','campaign_id'],how='left',validate='many_to_one')
        cp['missing_campaign']=cp.campaign_id.isna();cp['wrong_category']=~cp.missing_campaign & cp.category.ne(cp.target_item_group);cp['non_coupon_campaign']=~cp.missing_campaign & ~cp.coupon_flag.fillna(False).astype(bool)
        cp['discount_deviation']=(cp.discount_amount-cp.list_price_amount*cp.discount_rate).abs()
        cp['deviation_gt_half_yen']=~cp.missing_campaign & cp.discount_deviation.gt(.5000001)
        grouped=cp.groupby(['year','applied_coupon_campaign_id'],observed=True).agg(lines=('item_id','size'),list_sales=('list_price_amount','sum'),discount=('discount_amount','sum'),missing_campaign=('missing_campaign','sum'),wrong_category=('wrong_category','sum'),non_coupon_campaign=('non_coupon_campaign','sum'),deviation_gt_half_yen=('deviation_gt_half_yen','sum'),max_deviation=('discount_deviation','max')).reset_index();grouped['store_id']=sid;coupon_summaries.append(grouped)
        bad=cp[cp[['missing_campaign','wrong_category','non_coupon_campaign','deviation_gt_half_yen']].any(axis=1)].head(3)
        if len(bad):examples.append(bad.assign(store_id=sid))
        print(sid,'明細:',len(o),'クーポン適用:',len(cp),flush=True)
        del o,cp,p,c
checks=pd.concat(summaries,ignore_index=True);coupons=pd.concat(coupon_summaries,ignore_index=True)
print('価格・金額の検査（全店舗・全期間）');display(checks.groupby('store_id').sum(numeric_only=True).drop(columns='year'))
print('全件集計:');display(checks.drop(columns='year').select_dtypes('number').sum().to_frame('rows'))
print('クーポンに関する検査（全店舗・全期間）');display(coupons[['lines','missing_campaign','wrong_category','non_coupon_campaign','deviation_gt_half_yen']].sum().to_frame('rows'))
print('最大の値引き額と定価×割引率の絶対差:',coupons.max_deviation.max(),'円')
print('※0.5円基準は丸めの確認であり、配布データの端数処理ルールを断定しない。')
if examples: display(pd.concat(examples,ignore_index=True).head(20))
assert checks.lines.sum()==8417590

S001 明細: 1006670 クーポン適用: 12964
S002 明細: 866515 クーポン適用: 14221
S003 明細: 964313 クーポン適用: 32344
S004 明細: 1112487 クーポン適用: 39972
S005 明細: 801335 クーポン適用: 22536
S006 明細: 596023 クーポン適用: 10056
S007 明細: 731885 クーポン適用: 10336
S008 明細: 684457 クーポン適用: 16617
S009 明細: 579569 クーポン適用: 13816
S010 明細: 1074336 クーポン適用: 13757
価格・金額の検査（全店舗・全期間）
全件集計:
クーポンに関する検査（全店舗・全期間）
最大の値引き額と定価×割引率の絶対差: 0.5 円
※0.5円基準は丸めの確認であり、配布データの端数処理ルールを断定しない。


,lines,missing_price,missing_menu,bad_list_price,bad_net,discount_without_coupon,coupon_without_discount,negative_amount
store_id,,,,,,,,
S001,1006670,0,0,0,0,0,0,0
S002,866515,0,0,0,0,0,0,0
S003,964313,0,0,0,0,0,0,0
S004,1112487,0,0,0,0,0,0,0
S005,801335,0,0,0,0,0,0,0
S006,596023,0,0,0,0,0,0,0
S007,731885,0,0,0,0,0,0,0
S008,684457,0,0,0,0,0,0,0
S009,579569,0,0,0,0,0,0,0


,rows
lines,8417590
missing_price,0
missing_menu,0
bad_list_price,0
bad_net,0
discount_without_coupon,0
coupon_without_discount,0
negative_amount,0


,rows
lines,186619
missing_campaign,0
wrong_category,0
non_coupon_campaign,0
deviation_gt_half_yen,0


## キャンペーン別・年別の実際の適用
キャンペーンの予定日数ではなく、明細行に付いた実際の適用件数。割引率の平均は販売額で重み付けされた実効比率。

In [3]:
annual=coupons.groupby(['year','applied_coupon_campaign_id'],observed=True).agg(lines=('lines','sum'),list_sales=('list_sales','sum'),discount=('discount','sum'),max_deviation=('max_deviation','max')).reset_index()
annual['effective_discount_rate']=annual.discount/annual.list_sales
display(annual)
by_store=coupons.groupby('store_id',observed=True).agg(lines=('lines','sum'),list_sales=('list_sales','sum'),discount=('discount','sum'))
by_store['effective_discount_rate']=by_store.discount/by_store.list_sales
display(by_store)
print('クーポン適用明細総数:',int(coupons.lines.sum()),'全明細に占める割合:',float(coupons.lines.sum()/checks.lines.sum()))

クーポン適用明細総数: 186619 全明細に占める割合: 0.022170122327174406


,year,applied_coupon_campaign_id,lines,list_sales,discount,max_deviation,effective_discount_rate
0,2022,RULE_DESSERT_FAIR,13123,6666670,467408,0.5000,0.0701
1,2022,RULE_FAMILY_WEEKEND,14827,15111220,1511122,0.0000,0.1000
2,2022,RULE_KIDS_SMILE,4466,2936650,147550,0.5000,0.0502
3,2022,RULE_RAINY_DESSERT,18057,9074030,546203,0.4000,0.0602
4,2022,RULE_WEEKDAY_LUNCH,6645,5308490,424819,0.4000,0.0800
5,2023,RULE_DESSERT_FAIR,14234,7187130,503865,0.5000,0.0701
6,2023,RULE_FAMILY_WEEKEND,18518,18913630,1891363,0.0000,0.1000
7,2023,RULE_KIDS_SMILE,5086,3326520,167167,0.5000,0.0503
8,2023,RULE_RAINY_DESSERT,19429,9759070,587363,0.4000,0.0602
9,2023,RULE_WEEKDAY_LUNCH,6170,4891090,391382,0.4000,0.0800


,lines,list_sales,discount,effective_discount_rate
store_id,,,,
S001,12964,6840070,430806,0.0630
S002,14221,9445710,690078,0.0731
S003,32344,25979790,2279725,0.0877
S004,39972,31819190,2767641,0.0870
S005,22536,17875160,1551493,0.0868
S006,10056,5215340,331351,0.0635
S007,10336,5495410,336963,0.0613
S008,16617,11757260,886347,0.0754
S009,13816,10517880,888579,0.0845


## 解釈
- この照合は配布された過去データに関する整合性検査であり、予測対象の2025年に実際に適用されたクーポンや注文の有無を示さない。
- 適用予定と実際の利用は異なる。未来のクーポン実績や割引後単価を同日の説明変数に使わない。
- 明細の端数は行ごとの丸めや適用ルールに依存し得る。値引き額と定価×率の差だけで誤データと断定しない。